# UC-DGGS-1 — H3 Zone Aggregation via OGC API - DGGS

**Persona:** Spatial Data Scientist

**Goal:** Ingest point features (agricultural sample sites) via OGC Features
POST, then query `GET /dggs/catalogs/{cat}/collections/{col}/dggs?dggs-id=H3`
to retrieve the features aggregated into H3 hexagonal DGGS zones at a chosen
resolution level. Also list DGGRS definitions and inspect a single zone.

**Prerequisites:**
- GeoID platform running at `DYNASTORE_BASE_URL` (default `http://localhost:8080`)
- `dggs` and `features` extensions enabled in the active SCOPE
- Write-scoped JWT in `DYNASTORE_TOKEN` (or IDP client credentials set)

**References:**
- OGC API - DGGS 1.0 (OGC 20-039)
- Routes: `GET /dggs/dggs-list`, `GET /dggs/catalogs/{cat}/collections/{col}/dggs`,
  `GET /dggs/catalogs/{cat}/collections/{col}/dggs/{zoneId}`

In [ ]:
import json
import os

import httpx
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True), override=False)

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")

# Auto-provision token via IDP client_credentials when not already set
if not os.environ.get("DYNASTORE_TOKEN"):
    _idp = (os.environ.get("IDP_PUBLIC_URL") or os.environ.get("IDP_ISSUER_URL", "")).rstrip("/")
    _cid = os.environ.get("IDP_CLIENT_ID", "")
    _sec = os.environ.get("IDP_CLIENT_SECRET", "")
    if _idp and _cid and _sec:
        try:
            _r = httpx.post(
                f"{_idp}/protocol/openid-connect/token",
                data={"grant_type": "client_credentials", "client_id": _cid, "client_secret": _sec},
                timeout=10,
            )
            if _r.status_code == 200:
                os.environ["DYNASTORE_TOKEN"] = _r.json().get("access_token", "")
        except Exception:
            pass

TOKEN = (
    os.environ.get("DYNASTORE_TOKEN")
    or os.environ.get("DYNASTORE_WRITE_TOKEN")
    or os.environ.get("DYNASTORE_ADMIN_TOKEN")
    or ""
)

CATALOG_ID    = os.environ.get("CATALOG_ID", "demo-dggs")
COLLECTION_ID = os.environ.get("COLLECTION_ID", "agri-sites")
DGGS_ID       = os.environ.get("DGGS_ID", "H3")
ZONE_LEVEL    = int(os.environ.get("ZONE_LEVEL", "4"))  # H3 resolution 4 ~ 1000 km2 cells

headers = {"Authorization": f"Bearer {TOKEN}", "Content-Type": "application/json"}
client  = httpx.Client(base_url=BASE_URL, headers=headers, timeout=60.0)

print(f"Platform  : {BASE_URL}")
print(f"Catalog   : {CATALOG_ID}")
print(f"Collection: {COLLECTION_ID}")
print(f"DGGRS     : {DGGS_ID}  zone-level={ZONE_LEVEL}")

## Step 1 — List supported DGGRS

`GET /dggs/dggs-list` returns the Discrete Global Grid Reference Systems
supported by this deployment. A 200 confirms the DGGS extension is active.
This deployment supports H3 (Uber hexagonal) and S2 (Google spherical).

In [ ]:
r = client.get("/dggs/dggs-list")
print(f"status: {r.status_code}")

_dggs_active = r.status_code == 200

if _dggs_active:
    dggrs_list = r.json()
    grids = dggrs_list.get("dggrs", [])
    print(f"Supported DGGRS ({len(grids)}):")
    for g in grids:
        print(
            f"  id={g.get('id')}  title={g.get('title')}  "
            f"keywords={g.get('keywords', [])[:3]}"
        )
elif r.status_code == 404:
    print("SKIP: /dggs endpoint not found — dggs extension may not be active.")
else:
    print(r.text[:300])

## Step 2 — Create catalog and ingest point features

We create a demo catalog and an `agri-sites` collection, then POST point
features representing agricultural monitoring sites across central Italy.
Each site carries an `ndvi` property for zone-level aggregation.

In [ ]:
# --- catalog ---
r = client.post(
    "/features/catalogs",
    content=json.dumps({"id": CATALOG_ID, "title": "DGGS demo catalog"}),
)
print(f"catalog   : {r.status_code}", "(already exists)" if r.status_code == 409 else "")

# --- collection ---
r = client.post(
    f"/features/catalogs/{CATALOG_ID}/collections",
    content=json.dumps({"id": COLLECTION_ID, "title": "Agricultural monitoring sites"}),
)
print(f"collection: {r.status_code}", "(already exists)" if r.status_code == 409 else "")

# --- items: agri monitoring sites across central Italy ---
sites = [
    {"id": "site-01", "lon": 12.50, "lat": 41.90, "ndvi": 0.72, "crop": "wheat"},
    {"id": "site-02", "lon": 12.70, "lat": 42.10, "ndvi": 0.65, "crop": "olive"},
    {"id": "site-03", "lon": 11.80, "lat": 43.20, "ndvi": 0.81, "crop": "sunflower"},
    {"id": "site-04", "lon": 13.30, "lat": 43.50, "ndvi": 0.58, "crop": "barley"},
    {"id": "site-05", "lon": 12.30, "lat": 44.00, "ndvi": 0.69, "crop": "corn"},
    {"id": "site-06", "lon": 10.50, "lat": 43.70, "ndvi": 0.77, "crop": "vineyard"},
]

for site in sites:
    feat = {
        "type": "Feature",
        "id": site["id"],
        "geometry": {"type": "Point", "coordinates": [site["lon"], site["lat"]]},
        "properties": {"ndvi": site["ndvi"], "crop": site["crop"]},
    }
    r = client.post(
        f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/items",
        content=json.dumps(feat),
    )
    if r.status_code in (200, 201):
        print(f"  site '{site['id']}': created")
    elif r.status_code == 409:
        print(f"  site '{site['id']}': already exists")
    else:
        print(f"  site '{site['id']}': {r.status_code} {r.text[:200]}")

## Step 3 — Retrieve DGGS zone aggregation

`GET /dggs/catalogs/{cat}/collections/{col}/dggs?dggs-id=H3&zone-level=4`
returns a GeoJSON FeatureCollection where each feature represents an H3
hexagonal zone. The `zone_id`, `feature_count`, and aggregated property
statistics are carried in `properties`.

In [ ]:
if not _dggs_active:
    print("SKIP: DGGS extension not active.")
else:
    r = client.get(
        f"/dggs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/dggs",
        params={
            "dggs-id": DGGS_ID,
            "zone-level": ZONE_LEVEL,
            "bbox": "9.0,40.0,15.0,46.0",
            "parameter-name": "ndvi",
        },
    )
    print(f"DGGS zones status: {r.status_code}")

    _first_zone_id = None

    if r.status_code == 200:
        fc = r.json()
        zones = fc.get("features", [])
        print(f"Zones returned: {len(zones)}")
        for zone in zones:
            props = zone.get("properties", {})
            zone_id = props.get("zone_id", zone.get("id", ""))
            count   = props.get("feature_count", props.get("count", "?"))
            ndvi    = props.get("ndvi", props.get("ndvi_mean", "n/a"))
            print(f"  zone_id={zone_id}  feature_count={count}  ndvi={ndvi}")
            if _first_zone_id is None:
                _first_zone_id = zone_id
    elif r.status_code == 404:
        print("  404 — catalog or collection not found.")
    elif r.status_code == 422:
        print(f"  422 — {r.text[:300]}")
    else:
        print(r.text[:300])

## Step 4 — Inspect a single DGGS zone

`GET /dggs/catalogs/{cat}/collections/{col}/dggs/{zoneId}` returns the
details for one specific DGGS zone, including its geometry and the list
of features that fall within it.

In [ ]:
if not _dggs_active:
    print("SKIP: DGGS extension not active.")
elif not _first_zone_id:
    print("SKIP: no zone returned in step 3 — cannot inspect a zone.")
else:
    r = client.get(
        f"/dggs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/dggs/{_first_zone_id}",
        params={"dggs-id": DGGS_ID, "zone-level": ZONE_LEVEL},
    )
    print(f"Zone inspect status: {r.status_code}  zone={_first_zone_id}")

    if r.status_code == 200:
        zone_detail = r.json()
        props = zone_detail.get("properties", {})
        geom  = zone_detail.get("geometry", {})
        print(f"  zone_id     : {props.get('zone_id', zone_detail.get('id'))}")
        print(f"  geom type   : {geom.get('type') if geom else 'none'}")
        print(f"  feature_count: {props.get('feature_count', props.get('count', 'n/a'))}")
        features_in_zone = zone_detail.get("features", [])
        if features_in_zone:
            print(f"  features ({len(features_in_zone)}):")
            for f_ in features_in_zone[:5]:
                print(f"    id={f_.get('id')}")
    elif r.status_code == 404:
        print(f"  404 — zone '{_first_zone_id}' not found.")
    else:
        print(r.text[:300])

## Teardown

In [ ]:
_TEARDOWN = os.environ.get("DGGS_TEARDOWN", "true").lower() == "true"

if _TEARDOWN:
    r = client.delete(f"/features/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
    print(f"DELETE collection: {r.status_code}")
    r = client.delete(f"/features/catalogs/{CATALOG_ID}")
    print(f"DELETE catalog   : {r.status_code}")
else:
    print("Teardown skipped (set DGGS_TEARDOWN=true to enable).")

## Result — DGGS landing URL

The DGGS extension does not ship a dedicated web page. The canonical OGC
entry point is the DGGS landing page below.

In [ ]:
client.close()
print(f"DGGS landing page: {BASE_URL}/dggs/")
print(f"DGGRS list       : {BASE_URL}/dggs/dggs-list")
print(
    f"DGGS data URL    : "
    f"{BASE_URL}/dggs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/dggs"
    f"?dggs-id={DGGS_ID}&zone-level={ZONE_LEVEL}"
)